In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score

url = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "course_lead_scoring_2026.csv"
)

df = pd.read_csv(url)

print("Размер датасета:", df.shape)
print("\nПропуски до заполнения:")
print(df.isna().sum())

features = df.drop(columns="converted")

categorical = features.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numerical = features.select_dtypes(
    include="number"
).columns.tolist()

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

print("\nКатегориальные признаки:", categorical)
print("Числовые признаки:", numerical)

Размер датасета: (5000, 9)

Пропуски до заполнения:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

Категориальные признаки: ['lead_source', 'industry', 'employment_status', 'location']
Числовые признаки: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [2]:
print(df["industry"].value_counts())

print("\nОтвет Q1:", df["industry"].mode().iloc[0])

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

Ответ Q1: technology


In [3]:
correlation = df[numerical].corr()

display(correlation.round(3))

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

for first, second in pairs:
    score = correlation.loc[first, second]
    print(f"{first} + {second}: {score:.6f}")

best_pair = max(
    pairs,
    key=lambda pair: abs(correlation.loc[pair[0], pair[1]])
)

print("\nОтвет Q2:", best_pair)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000,0.161,0.123,0.229
number_of_courses_viewed,0.161,1.000,0.722,0.757
interaction_count,0.123,0.722,1.000,0.916
lead_score,0.229,0.757,0.916,1.000


interaction_count + lead_score: 0.915746
number_of_courses_viewed + lead_score: 0.757204
number_of_courses_viewed + interaction_count: 0.721609
annual_income + interaction_count: 0.122842

Ответ Q2: ('interaction_count', 'lead_score')


In [4]:
df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
)

df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=42,
)

y_train = df_train.pop("converted").to_numpy()
y_val = df_val.pop("converted").to_numpy()
y_test = df_test.pop("converted").to_numpy()

print("Train:", len(df_train))
print("Validation:", len(df_val))
print("Test:", len(df_test))

Train: 3000
Validation: 1000
Test: 1000


In [5]:
mi_scores = {}

for column in categorical:
    score = mutual_info_score(y_train, df_train[column])
    mi_scores[column] = score

    print(f"{column}: {round(score, 2):.2f}")

print(
    "\nОтвет Q3:",
    max(mi_scores, key=lambda column: round(mi_scores[column], 2))
)

lead_source: 0.03
industry: 0.00
employment_status: 0.02
location: 0.00

Ответ Q3: lead_source


In [6]:
def train_and_evaluate(columns, C=1.0):
    train_dicts = df_train[columns].to_dict(orient="records")
    val_dicts = df_val[columns].to_dict(orient="records")
    dv = DictVectorizer(sparse=False)

    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42,
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    return accuracy_score(y_val, y_pred)


all_features = df_train.columns.tolist()

original_accuracy = train_and_evaluate(all_features)

print("Accuracy:", original_accuracy)
print("Ответ Q4:", round(original_accuracy, 2))

Accuracy: 0.645
Ответ Q4: 0.65


In [7]:
candidates = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count",
]

differences = {}

print("Исходная accuracy:", original_accuracy)

for removed_feature in candidates:
    remaining_features = [
        feature
        for feature in all_features
        if feature != removed_feature
    ]

    accuracy_without = train_and_evaluate(remaining_features)

    difference = original_accuracy - accuracy_without
    differences[removed_feature] = difference

    print(
        f"Без {removed_feature}: "
        f"accuracy={accuracy_without:.3f}, "
        f"difference={difference:.3f}"
    )

print("\nОтвет Q5:", min(differences, key=differences.get))

Исходная accuracy: 0.645
Без lead_source: accuracy=0.642, difference=0.003
Без number_of_courses_viewed: accuracy=0.643, difference=0.002
Без interaction_count: accuracy=0.601, difference=0.044

Ответ Q5: number_of_courses_viewed


In [8]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_scores = {}

for C in c_values:
    accuracy = train_and_evaluate(all_features, C=C)
    c_scores[C] = round(accuracy, 3)

    print(f"C={C:.6f}: accuracy={c_scores[C]:.3f}")

best_C = min(
    c_values,
    key=lambda C: (-c_scores[C], C)
)

print("\nОтвет Q6:", best_C)

C=0.000001: accuracy=0.598
C=0.000010: accuracy=0.598
C=0.000100: accuracy=0.613
C=0.001000: accuracy=0.645

Ответ Q6: 0.001
